# Sparse Walker temporal skips — verified launcher

Fresh single-cell launcher. It imports directly from source, exercises the exact `encode` + FullCE training path, then launches the ML-1M temporal-skip experiment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil, torch
REPO='/content/Sparsewalker'
BRANCH='agent/walker-temporal-skips'
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
SRC=f'{REPO}/src'
sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + (':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')

import sparsewalker
print('sparsewalker from', sparsewalker.__file__)
from sparsewalker.models import SparseWalkerTemporalMemory
from sparsewalker.models.core import ar_training_loss
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU', torch.cuda.get_device_name(0), 'bf16', torch.cuda.is_bf16_supported())

# Smoke test the exact APIs used by the real trainer: encode() + FullCE loss.
m=SparseWalkerTemporalMemory(3706,200,d=64,layers=2,side=256,h=16,active=8,top_side=2,degree=4).cuda().train()
x=torch.randint(1,3707,(4,41),device='cuda')
with torch.autocast('cuda',dtype=torch.bfloat16):
    H=m.encode(x[:,:-1])
    loss=ar_training_loss(m,x,loss_mode='full')
loss.backward()
print('SMOKE OK', tuple(H.shape), 'loss', float(loss.detach()))
del m,x,H,loss
torch.cuda.empty_cache()

cmd=[sys.executable, f'{REPO}/experiments/run_ml1m_temporal_skips.py',
     '--seed','42','--max-epochs','20','--eval-every','2','--patience','8',
     '--batch-size','128','--eval-batch-size','1024']
print('RUNNING', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO, env=os.environ.copy(), check=True)


In [ ]:
import pandas as pd
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_temporal_skips/ml1m/seed42/history.csv')
if p.exists():
    df=pd.read_csv(p)
    display(df[['epoch','loss','NDCG@10','HR@10','MRR@10','memory_share','seconds','positions_per_s']])
    print('best temporal val NDCG@10', df['NDCG@10'].max())
else:
    print('No history yet')
